# Chapter 4: Risk and Compliance — Credit Risk Fundamentals
**Financial AI in Practice** | Manning Publications

This notebook accompanies section 4.4.3, "Weight of Evidence and information value." Listing 4.3 takes one binned
variable, `AgeGroup`, with counts of good (non-default) and bad (default) borrowers in each bin, and computes each
bin's Weight of Evidence (the log of the bin's share of bads over its share of goods, so positive values mark bins
riskier than the population) and the variable's information value, the sum of `(BadRate − GoodRate) × WoE` over the
bins. The small table is defined inline in the listing, so no data file is needed. As the book discusses on p. 86,
the result (IV ≈ 0.7582) is above the 0.5 "suspiciously strong" mark, which in a real project would prompt a check for
data leakage before the variable goes into a scorecard.

### Listing 4.3 — Understand WoE and IV with data
*Section 4.4.3, Short Python example (p. 85; the book's output is figure 4.8)*

The book's margin notes on this listing:

- `data = {...}`: defines sample data for `AgeGroup` bins, along with counts of good vs. bad borrowers.
- `total_good`, `total_bad`: aggregate the total number of Good and Bad observations across all age groups.
- `GoodRate`, `BadRate`: compute each bin's proportion of Good (`GoodRate`) and Bad (`BadRate`).
- `df_bins['WoE']`: calculates the WoE (Weight of Evidence) for each bin.
- `IV_age`: sums up each bin's contribution (`IV_bin`) to produce a single IV for "AgeGroup."

In [1]:
import numpy as np
import pandas as pd

data = {
    'AgeGroup':   ['<20', '20-24', '25-29', '30-39', '40+'],
    'GoodCount':  [90, 120, 160, 300, 280],
    'BadCount':   [10,  30,  40,   25,   5]
}

df_bins = pd.DataFrame(data)

total_good = df_bins['GoodCount'].sum()
total_bad  = df_bins['BadCount'].sum()

df_bins['GoodRate'] = df_bins['GoodCount'] / total_good
df_bins['BadRate']  = df_bins['BadCount']  / total_bad

df_bins['WoE'] = np.log(
    (df_bins['BadRate'] + 1e-9) / (df_bins['GoodRate'] + 1e-9)
)

df_bins['IV_bin'] = (
    (df_bins['BadRate'] - df_bins['GoodRate']) * df_bins['WoE']
)
IV_age = df_bins['IV_bin'].sum()

print(df_bins[['AgeGroup','GoodCount','BadCount','WoE','IV_bin']])
print(f"Information Value (Age) = {IV_age:.4f}")

  AgeGroup  GoodCount  BadCount       WoE    IV_bin
0      <20         90        10 -0.041243  0.000158
1    20-24        120        30  0.769687  0.112691
2    25-29        160        40  0.769687  0.150255
3    30-39        300        25 -0.328925  0.029115
4      40+        280         5 -1.869370  0.466001
Information Value (Age) = 0.7582
